# 03c. Practice Set 2

**No new material:** practice on notebooks 02, 03a and 03b, mostly on one dataset (Titanic):

- **Part A**, warm-up: regularization by hand.
- **Part B**, the full workflow: look, split, scale, fit, evaluate, interpret.
- **Part C**, choosing a threshold by cost, the right way (on validation data).
- **Part D**, choosing `C`, the right way.
- **Part E**, your own implementation vs. sklearn, once more.
- **Part F**, predict, then run: short experiments where you must **guess the result first**.

Less guidance than before: you'll write most of the code yourself. Reuse code from earlier notebooks
freely; that's what it's there for.

## 0. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)
plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

# Interactive sliders (optional): if ipywidgets is missing, we show a few fixed pictures instead.
try:
    from ipywidgets import interact, FloatSlider, FloatLogSlider
    HAVE_WIDGETS = True
except ImportError:
    HAVE_WIDGETS = False
    print("ipywidgets is not installed: the interactive cells will show fixed pictures instead.")

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score

def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

In [ ]:
# --- exercise checker: just run this cell -----------------------------------------------
import base64 as _b64, json as _json
_ANS = _json.loads(_b64.b64decode("eyJBMSI6IDAuMDAxMjUsICJBMiI6IFswLjg4MDc5NzA3Nzk3Nzg4MjMsIDEuOTExOTIwMjkyMjAyMjExOF0sICJBMyI6ICJhIiwgIkIxIjogWzAuNzQ0MDI3MzAzNzU0MjY2MiwgMC4xOTIzMDc2OTIzMDc2OTIzMl0sICJCMiI6IFs2MjksIDIxMF0sICJCMyI6IFswLjgsIDAuODUzNTI2NjUzMjY4MjU1MywgNTksIDIwLCAyMiwgMTA5XSwgIkI0IjogWyJTZXhfbWFsZSIsICJQY2xhc3MiXSwgIkMxIjogMC4zMzMzMzMzMzMzMzMzMzMzfQ=="))

def _has_str(v):
    return isinstance(v, str) or (isinstance(v, (list, tuple)) and any(_has_str(x) for x in v))

def _norm(v):
    return [_norm(x) for x in v] if isinstance(v, (list, tuple, np.ndarray)) else str(v).strip().lower()

def _close(a, b, tol):
    if _has_str(b):
        return _norm(a) == _norm(b)
    try:
        return np.allclose(np.asarray(a, dtype=float), np.asarray(b, dtype=float), atol=tol, rtol=0)
    except Exception:
        return False

def check(key, value, tol=1e-3):
    # Compare your answer for exercise `key` with the expected one.
    if value is None or value is Ellipsis or (isinstance(value, (list, tuple)) and any(v is None for v in value)):
        print(f"⏳ {key}: not answered yet")
    elif _close(value, _ANS[key], tol):
        print(f"✅ {key}: correct!")
    else:
        print(f"❌ {key}: not yet. You got {value}; check your work and try again")

def check_against(key, value, reference, tol=1e-3):
    # Compare your answer with a reference value computed in the same cell.
    if value is None or (isinstance(value, (list, tuple)) and any(v is None for v in value)):
        print(f"⏳ {key}: not answered yet")
    elif _close(value, reference, tol):
        print(f"✅ {key}: correct!")
    else:
        print(f"❌ {key}: not yet. You got {value}; check your work and try again")

def check_fn(key, fn, tol=1e-3):
    # Call your function on some test inputs and compare with the expected outputs.
    spec = _ANS[key]
    try:
        outs = [fn(*[np.array(a, dtype=float) for a in args]) for args in spec["args"]]
    except Exception as e:
        print(f"❌ {key}: your function raised an error: {type(e).__name__}: {e}")
        return
    if any(o is None for o in outs):
        print(f"⏳ {key}: not answered yet (your function returns None)")
    elif all(_close(o, ex, tol) for o, ex in zip(outs, spec["out"])):
        print(f"✅ {key}: correct!")
    else:
        print(f"❌ {key}: not yet. Your function gives {outs}; check your work and try again")

## Part A: regularization by hand

**A1.** A training set has $N = 400$ examples, and we use `LogisticRegression(C=2)`. What is the
corresponding $\lambda$ in our formula $\text{log-loss} + \frac{\lambda}{2}\|w\|^2$? Store it in `a1`.

In [ ]:
# YOUR CODE HERE
a1 = None
check("A1", a1, tol=1e-6)

**A2.** A model with **one** feature has $w = 2$ and $b = 0$. There is a single training example,
$x = 1$, $y = 1$. We use $\lambda = 0.5$ and $\eta = 0.1$.

1. Compute the gradient of the **regularized** loss with respect to $w$: $(h - y)\,x + \lambda w$.
2. Compute the new $w$ after one gradient-descent step.

Store them in `a2 = [gradient, new_w]`. Then, in words: the example is predicted **correctly** (with
$h = \sigma(2) \approx 0.88$), yet $w$ went **down**. Why?

In [ ]:
# YOUR CODE HERE
a2 = None
print(a2)
check("A2", a2)

**Your answer:** *(write here)*

**A3.** Three models are trained on the same (scaled) data: (a) `C=0.01`, (b) `C=1`, (c) `C=100`.
Which one gives the **softest** probabilities (closest to the overall fraction of positives)? Store the
letter in `a3`.

In [ ]:
# YOUR CODE HERE
a3 = None
check("A3", a3)

## Part B: the full workflow, on Titanic

The famous dataset of the Titanic's passengers: who survived the sinking in 1912? Each row is one
passenger:

- `Pclass`: ticket class, 1 to 3 (1st class is the most expensive);
- `Sex`;
- `Age`;
- `SibSp`: the number of siblings or spouses aboard;
- `Parch`: the number of parents or children aboard;
- `Fare`: the ticket price;
- `Embarked`: the port where they boarded;
- `Survived`: 1 if they survived.

We'll use the numeric features, plus `Sex` coded as 0/1. (`Embarked` is a *category* with three values;
we'll see how to handle categories in notebook 04.)

In [ ]:
titanic = pd.read_csv("../data/titanic.csv")
titanic = titanic.dropna(subset=["Fare"])                 # one incomplete row
titanic.head()

### B1. Look

Using pandas, compute the survival rate of **women** and of **men**. Store them in
`b1 = [women_rate, men_rate]`.

*Reminder:* `df[df["col"] == value]` selects rows, and `.mean()` of a 0/1 column is the fraction of 1s.
(Or, the short way: `df.groupby("col")["other"].mean()`.)

Then also look at the survival rate per `Pclass`. What do you expect the **signs** of the weights of
`Sex_male` and `Pclass` to be?

In [ ]:
# YOUR CODE HERE
b1 = None
print(b1)
check("B1", b1)

**Your answer:** *(write here)*

### B2. Prepare, split and scale

The features are prepared for you in the next cell. Then **you**:

1. split `X_t, y_t` into a training part and a test part: 25% test, stratified, `random_state=0`. Name
   them `X_train, X_test, y_train, y_test`, and store their sizes in `b2 = [n_train, n_test]`;
2. scale them with a `StandardScaler` fitted on the **training** part only. Name the scaled versions
   `X_train_s, X_test_s`.

In [ ]:
X_df = titanic[["Pclass", "Age", "SibSp", "Parch", "Fare"]].copy()
X_df["Age"] = X_df["Age"].fillna(X_df["Age"].median())    # fill in the missing ages
X_df["Sex_male"] = (titanic["Sex"] == "male").astype(int)  # 1 = male, 0 = female
feature_names = list(X_df.columns)
X_t = X_df.values.astype(float)
y_t = titanic["Survived"].values
print("X:", X_t.shape, "  features:", feature_names)

In [ ]:
# YOUR CODE HERE
b2 = None
check("B2", b2)

### B3. Fit and evaluate

Fit a `LogisticRegression` (default settings) on the scaled training data, and call it `model`. On the
**test** set, compute the accuracy, the AUC, and the confusion counts. Store
`b3 = [accuracy, auc, TP, FP, FN, TN]`, and print the `classification_report`.

In [ ]:
# YOUR CODE HERE
b3 = None
print(b3)
check("B3", b3, tol=0.01)

### B4. Interpret

Print the weights next to the feature names. Which **two** features have the strongest effect (largest
absolute weight)? Store their names in `b4`, strongest first. Do the signs match your expectations from
B1? What does the weight of `Age` say?

In [ ]:
# YOUR CODE HERE
b4 = None
print(b4)
check("B4", b4)

**Your answer:** *(write here)*

## Part C: choosing a threshold by cost, the right way

Suppose (as a thought experiment) that a **missed survivor** (predicting "died" for someone who
survived, a false negative) costs **2**, and a **false alarm** (a false positive) costs **1**.

**C1.** Using the formula from 01c, what's the best threshold? Store it in `c1`.

In [ ]:
# YOUR CODE HERE
c1 = None
check("C1", c1)

**C2.** The formula assumes that the model's probabilities are accurate. Let's check it on data.
But **not** on the test set: choosing the threshold is a decision, just like choosing `C`, so it must be
made on **validation** data (03b, section 5.3).

1. Split the **training** data again: 30% validation, stratified, `random_state=0`. Scale (fit on the
   new, smaller training part), and fit a model on it.
2. For every threshold in `np.arange(0.05, 0.96, 0.05)`, compute the total cost on the **validation**
   part. Plot it.
3. Which threshold is cheapest? Is it close to the formula's answer?
4. Now, and only now, use that threshold on the **test** set once, and report its cost there.

In [ ]:
# YOUR CODE HERE

**Your answer:** *(write here)*

## Part D: choosing `C`, the right way

**D1** (predict first). Titanic has $N \approx 630$ training passengers and only $n = 6$ features.
Compare that with Ionosphere in 03b (about 160 training examples and 33 features). Before running
anything: do you expect `C` to matter **a lot** or **a little** here? Why?

**D2.** Now check. For $C \in \{0.001, 0.01, 0.1, 1, 10, 100, 1000\}$, average the training and
validation **AUC** over 10 random training/validation splits of the training data (30% validation),
scaling inside each split. Plot both against `C`. Which `C` would you choose?

In [ ]:
# YOUR CODE HERE

**Your answer:** *(write here)*

## Part E: your own implementation, once more

**E1.** Write `fit_reg(X, y, eta, n_iter, lam)`, gradient descent with L2 regularization, **from
memory** if you can (look back at 03b only if you're stuck). It should return `w, b`.

**E2.** Train it on the scaled Titanic training data with the $\lambda$ that corresponds to `C=1`
($\lambda = \frac{1}{C N}$), using `eta=0.5` and 5000 steps. Compare its weights with `model.coef_[0]`
from B3. The largest difference should be below 0.01.

In [ ]:
def fit_reg(X, y, eta, n_iter, lam=0.0):
    # YOUR CODE HERE
    return None, None

In [ ]:
# YOUR CODE HERE

## Part F: predict, then run

For each experiment, **write your prediction first** (in the markdown cell), then run the code and
compare. Being wrong is fine; being surprised is how you learn.

**F1.** Train on the scaled Titanic data with `C=0.001`. Predict: what will the weights look like? What
will the range of predicted probabilities on the test set be (smallest and largest)?

*My prediction:*

In [ ]:
# YOUR CODE HERE

**Your answer:** *(write here)*

**F2.** Run your `fit_reg` on the **unscaled** training data `X_train` (with `lam=0`), with
`eta=0.5` and 2000 steps. Predict: what will happen? (Look at the ranges of the features: `Fare` goes up
to more than 500.)

*My prediction:*

In [ ]:
# YOUR CODE HERE

**Your answer:** *(write here)*

**F3.** Remove the feature `Sex_male` (the last column), retrain with default settings, and
compute the test AUC. Predict first: will the AUC drop a little (say, from 0.85 to 0.83) or a lot?

*My prediction:*

In [ ]:
# YOUR CODE HERE

**Your answer:** *(write here)*

**F4.** Last one: double every feature of the (scaled) training and test data (`2 * X_train_s`),
and retrain with the default `C=1`. Predict: will the test AUC change? Will the weights change? (Think
about 01b, exercise 5.B, and about 03b, section 6.)

*My prediction:*

In [ ]:
# YOUR CODE HERE

**Your answer:** *(write here)*

## References and further reading

- For review: MLU-Explain's
  [Train, Test, and Validation Sets](https://mlu-explain.github.io/train-test-validation/) and
  [The Bias-Variance Tradeoff](https://mlu-explain.github.io/bias-variance/).
- pandas: [Group by: split-apply-combine](https://pandas.pydata.org/docs/user_guide/groupby.html), used
  in B1.